<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_Simulation_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

## Middle East Oil Security Simulation Engine
MOS is an LLM-driven geopolitical simulation in which strategic actors choose actions within an explicitly represented and evolving WorldState. Chanakya proposes actor-specific strategic options, Centaur evaluates their relative likelihood, and ZeitWorld translates the selected Event into the next WorldState. The objective is not to predict the future, but to explore plausible, path-dependent trajectories arising from alternative strategic choices.

In [ ]:
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py
!wget -O mosCentaur.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosCentaur.py

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *

In [ ]:
import importlib

import mos
import mosZeitWorld
import mosChanakya
import mosCentaur

importlib.reload(mos)
importlib.reload(mosZeitWorld)
importlib.reload(mosChanakya)
importlib.reload(mosCentaur)

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *

## Initialise Runtime Environment

This cell prepares the runtime environment required by the MOS simulation.

It performs three functions:

1. **Initialises the OpenAI connection**
   - Authenticates the OpenAI client using `authenticateOpenAI()`.
   - Selects `gpt-5.6-luna` as the working LLM for the simulation.

2. **Mounts the MOS persistent data store**
   - Mounts Google Drive in the Colab runtime.
   - Defines `MOS_HOME` as the root directory of the MOS project.
   - Establishes paths to the major persistent components:
     - `WorldFacts` — real-world evidence used to initialise the simulation.
     - `ZeitWorld` — roles and prompts governing WorldState creation and evolution.
     - `Chanakya` — actor roles and strategy prompts used to generate Candidate Events.
     - `Centaur` — adjudication roles used to assign relative simulation weights to Candidate Events.

3. **Verifies the project directories**
   - Lists the contents of the WorldFacts, ZeitWorld, Chanakya and Centaur directories.
   - Separately displays the static `MD` evidence and dynamic `NEWS` evidence directories.
   - This provides a quick runtime check that the expected Google Drive resources are available before the simulation begins.

The Google Drive directories contain persistent evidence, roles and prompts. Runtime simulation objects such as WorldStates, Events, Candidate Events and Centaur weights are created subsequently in memory.

In [ ]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

# Real-world evidence, LLM roles and prompts
# are stored persistently in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"
CT_PATH = f"{MOS_HOME}/Centaur"

!ls -l "{WORLDFACTS_DIR}"
!ls -lh "{WORLDFACTS_DIR}/MD"
!ls -lh "{WORLDFACTS_DIR}/NEWS"
!ls -lh "{ZW_PATH}"
!ls -lh "{CH_PATH}"
!ls -lh "{CT_PATH}"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com
Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF
total 163K
-rw------- 1 root root 7.5K Oct  2 07:03 IranData_261002.md
-rw------- 1 root root 5.6K Oct  2 07:03 MissilesofIranMissil_261002.md
-rw------- 1 root root 8.3K Oct  2 07:04 OilMarketReportSepte_261002.md
-rw------- 1 root root  12K Oct  2 07:04 PressReleasePagePres_261002.md
-rw------- 1 root root 4.3K Oct  2 07:04 ShortTermEnergyOutlo_261002.md
-rw------- 1 root root  37K Oct  2 07:06 SnapshotofIndiasOilG_261002.md
-rw------- 1 root root  18K Oct  2 07:06 UKCommonsIranin2026S_261002.md
-rw------- 1 root root  70K Oct  2 07:08 USCongressR488873_261002.md
total 71K
-rw------- 1 root root 71K Oct  2 07:14 MOS-MC-LiveFeed_261002_1244_IST.txt
total 13K
-rw------- 1 root root 2.2K Oct  1 00:10 ZW_Initialisation_Prompt_v0.txt
-rw-----

## Gathering World Facts

In [ ]:
# ------------------------------------------------------------
# Build WorldFactBase
#
# Static PDF-derived Markdown files and live Moneycontrol news
# are assembled into a single corpus used to initialise W0.
# No real-world information enters the simulation after W0.
# ------------------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)

Static sources : 8
Live sources   : 1
Characters     : 235,072
Words          : 35,600


### Defining the World

In [ ]:
# Define the WorldState variables
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------

worldVariables = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [ ]:
# Initialise WorldState history
#
# W stores the evolving WorldStates of the simulation.
# W0_Template defines the empty structure from which W0 is created.
# ------------------------------------------------------------

W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

## Agents and their Roles

### ZeitWorld

In [ ]:
# Load ZeitWorld roles and prompts
#
# ZW_Role                  : defines ZeitWorld's role and discipline
# ZW_Initialisation_Prompt : creates W0 from the WorldFactBase
# ZW_Update_Prompt         : evolves the current WorldState after an Event
# ------------------------------------------------------------

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")
ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [ ]:
# Instantiate ZeitWorld
#
# ZeitWorld is created with its LLM client, model and governing role.
# The initialised flag confirms whether W0 has already been created.
# ------------------------------------------------------------

ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
print(ZeitWorld.initialised)
#print(ZeitWorld.model)

False


### Chanakya

In [ ]:
# Load Chanakya roles and strategy
#
# Generic Role : defines the common strategic discipline of all Chanakyas
# US Role      : defines US objectives, interests and constraints
# Iran Role    : defines Iranian objectives, interests and constraints
# BinaryHD     : generates two contrasting Hawk and Dove candidates
# ============================================================

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")
CH_BinaryHD     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")

In [ ]:
# Instantiate Chanakya agents
#
# Each Chanakya represents a national actor with its own
# interests and constraints, using the current Binary Hawk/Dove strategy.
# The registry maps each actor to its corresponding Chanakya.
# ============================================================

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_BinaryHD
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_BinaryHD
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


### Centaur

In [ ]:
# Load Centaur roles
#
# Generic Role : defines the common adjudication discipline of all Centaurs
# UNSG Role    : provides a fair, independent UN diplomatic perspective
# ============================================================


CT_Generic_Role = getPrompt(f"{CT_PATH}/CT_Generic_Role_v0.txt")
CT_UNSG_Role      = getPrompt(f"{CT_PATH}/CT_UNSG_Role_v0.txt")

In [ ]:
# Instantiate Centaur adjudicator
#
# CentaurUNSG evaluates Chanakya's Candidate Events from the
# perspective of a fair and independent UN Secretary-General.
# ============================================================
Centaur_UNSG = Centaur_Agent(
    agent_id="CentaurUNSG",
    generic_role=CT_Generic_Role,
    adjudication_role=CT_UNSG_Role
)

## Initialisation of Scenario

In [ ]:
# Initialise the first WorldState
#
# ZeitWorld uses the WorldFactBase and W0 template to create W0.
# The completed WorldState is automatically appended to W as W[0].
# ============================================================

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 57,064 | Output: 1,934 | Reasoning: 132 | Total: 58,998
True


In [ ]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

In [ ]:
# Define initial Trigger options
#
# Two alternative Events are available to start the simulation.
# One is manually selected as E0; thereafter Events are generated
# and selected by the simulation.
# ============================================================

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


In [ ]:
# Select the initial Trigger
#
# The manually selected DefaultTrigger starts the first epoch.
# Its selection weight is None because it was not chosen by Shakuni.
# ============================================================
DefaultTrigger = Trigger1
#Trigger = Trigger2
DefaultTrigger.show()
print(DefaultTrigger.selection_weight)


Event     : 
Actor     : Iran
Action    : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
Rationale : Initial Trigger for Iran
selection_weight : None
None


In [ ]:
# ============================================================
# Initialise / Reset Scenario
#
# Retains W0 and clears all subsequent WorldStates and Events,
# restoring the simulation to its starting condition.
# ============================================================

def resetWorld():
    global E, ActiveChanakya, Candidates

    del W[1:]
    E = []

    ActiveChanakya = None
    Candidates = []

    print("Scenario reset to W0.")

# ------------------------------------------------------------

resetWorld()

## Single Epoch

### Define Epoch

In [ ]:
def runEpoch():
    # ============================================================
    # EPOCH STEP 1 — Establish Current Trigger
    #
    # The epoch is derived from the current WorldState history.
    # For the first epoch, DefaultTrigger becomes E0.
    # Thereafter, the last Reaction becomes the next CurrentTrigger.
    # ============================================================

    epoch = len(W) - 1
    if len(E) == 0:
        CurrentTrigger = DefaultTrigger
        CurrentTrigger.event_id = f"E{epoch}"
        E.append(CurrentTrigger)
    else:
        CurrentTrigger = E[-1]

    E[epoch].show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 1 complete" )

    # ============================================================
    # EPOCH STEP 2 — Update WorldState
    #
    # ZeitWorld applies Trigger E[epoch] to W[epoch] and creates
    # the next WorldState W[epoch+1].
    # ============================================================

    _ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

    E[epoch].show()
    W[epoch+1].show(-2)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 2 complete" )

    # ============================================================
    # EPOCH STEP 3 — Select Active Actor
    #
    # The Turn Controller identifies which actor responds to the
    # Current Trigger and activates its corresponding Chanakya.
    # ============================================================

    ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)
    ActiveChanakya.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 3 complete" )

    # ============================================================
    # EPOCH STEP 4 — Generate Candidate Reactions
    #
    # The Active Chanakya evaluates the updated WorldState and
    # generates Candidate Events as possible reactions to the Trigger.
    # ============================================================

    Candidates = ActiveChanakya.propose(
        client=client,
        model=model,
        trigger=E[epoch],
        world=W[epoch+1]
    )

    for i, candidate in enumerate(Candidates):
        print(f"\nCandidate Reaction {i+1}")
        print("-" * 60)
        candidate.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 4 complete" )

    # ============================================================
    # EPOCH STEP 5 — Adjudicate Candidate Reactions
    #
    # Centaur evaluates the Candidate Events against the current
    # WorldState and assigns each a relative simulation weight.
    # ============================================================

    weights = Centaur_UNSG.evaluate(
        client,
        model,
        CurrentTrigger,
        W[epoch + 1],
        Candidates
    )

    print("\nCentaur Weights")
    print("----------------")
    print(weights)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 5 complete" )

    # ============================================================
    # EPOCH STEP 6 — Select Reaction
    #
    # Shakuni makes a single stochastic selection from the Candidate
    # Events using Centaur's relative weights. The selected Reaction
    # records its weight, receives the next Event ID, and is appended
    # to E, becoming the Trigger for the next epoch.
    # ============================================================

    Reaction = Shakuni(Candidates, weights)

    Reaction.event_id = f"E{len(E)}"
    E.append(Reaction)

    print("\nShakuni has spoken")
    print("------------------")
    Reaction.show()
    #print(Reaction.selection_weight)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 6 (last) complete" )



### Run Epoch

In [ ]:
runEpoch()

Event     : E1
Actor     : US
Action    : Expand military operations to suppress Iranian maritime, missile, and drone capabilities threatening US forces and tanker transit, while intensifying enforcement of the blockade and organizing assertive allied convoy operations through the Strait of Hormuz.
Rationale : A forceful response could restore deterrence after direct attacks on US naval forces, impose costs on Iran for attempting to close the Strait, and demonstrate that commercial access will not be determined solely by Iranian coercion. Coordinated action with regional and allied partners could also reinforce maritime security and preserve coalition credibility.
selection_weight : 0.72
epoch : 1 len(W): 2 len(E): 2  ***** Step # 1 complete
Updating WorldState ---------- W1
Model: gpt-5.6-luna | Effort: medium | Input: 3,388 | Output: 2,561 | Reasoning: 279 | Total: 5,949
Appended  WorldState ---------- W2
Event     : E1
Actor     : US
Action    : Expand military operations to suppres

In [ ]:
#print(W)

In [ ]:
def resetWorld():
    global ActiveChanakya, Candidates
    del W[1:]
    E.clear()
    ActiveChanakya = None
    Candidates = []

    print("World reset to W0.")
# ------------------------
resetWorld()

World reset to W0.


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)